## Award B Submission - SEAS the Moment

**Team info**
| Legal name | Affiliation | Institutional email | Kaggle username |
|---|---|---|---|
| Jasmine Andresol | Harvard SEAS | jasmineandresol@college.harvard.edu | jasmineandresol |
| Eddy Kang | Harvard SEAS | eddykang@college.harvard.edu | eddykang06 |
| Lenny Pische | Harvard SEAS | lenny_pische@college.harvard.edu | lpiske |
| William Liu | Harvard SEAS | wmliu@college.harvard.edu | wmliu1 |

**Registered team name:** SEAS the Moment

**GitHub repository:** https://github.com/lennardpische/staix26_seasthemoment

**Architecture - Mixture of Experts**

| Component | What it does |
|---|---|
| Experts (4) | Four independent specialist models, one per `experts/<name>/`, each trained on its own feature view: **Lenny** - multimodal FT-Transformer (tabular + mpnet text + EfficientNet image embeddings, resumes from `checkpoints/`); **William** - classical-statistics temporal model (`pipeline_v12.py`); **Jasmine** - healthcare-feature LightGBM; **Eddy** - XGBoost tree ensemble. |
| Isolation | Each expert runs in its **own subprocess** (cwd = repo root) so their identically-named modules (`features.py`, `models.py`, …) never collide; each writes `expert_<name>.csv` with columns `row_id`, `rate_per_10000_ed_visits`. |
| Gate (combine) | Per scoring category (`all_drugs`, `all_opioids`, `all_stimulants`): an **inverse-MAE weighted average** of the experts, weight ∝ 1 / (that expert's cross-validated Out of Fold Mean Absolute Error for the category), then `clip(lower=0)`. Weights are a **fixed `OOF_MAES` table** measured offline - there is no learned router and nothing is parsed from stdout at runtime. |
| Constraint | Post-hoc **nesting**: `all_drugs = max(all_drugs, all_opioids, all_stimulants)` per row, so the aggregate rate never falls below its components. |
| Output | `submission.csv` rebuilt from the template `row_id`s - schema-exact, 0 NaN, non-negative. |

---
### Inputs (attach all three; internet off)
- **Competition data**, auto-detected under `/kaggle/input/`.
- **`staix-pretrained-models`**: mpnet and EfficientNet weights used to build Lenny's text and image embeddings.
- **`seasthemoment-staix26`**: the repo code plus the trained artifacts (`embeddings/`, `checkpoints/`, `expert_*.csv`).

### How to run
Run all cells top to bottom. The notebook copies the code and artifacts into `/kaggle/working`, reuses each shipped `expert_<name>.csv` instead of retraining, combines them, and writes `submission.csv`. Every expert reads `train/`, `val/`, and `sample_submission.csv` at runtime and takes its `row_id`s from the template, so no period_ids or row counts are hardcoded.

## 1. Setup - load code + artifacts from attached datasets

In [ ]:
import shutil, os
from pathlib import Path

# Find the attached repo-code dataset wherever Kaggle mounted it (any depth).
src = next((d.parent for d in Path('/kaggle/input').rglob('experts') if d.is_dir()), None)
assert src is not None, ('Code dataset not attached. /kaggle/input tree top: '
                         + str([str(p) for p in Path('/kaggle/input').glob('*')]))
shutil.copytree(str(src), '/kaggle/working', dirs_exist_ok=True)
os.chdir('/kaggle/working')
import torch
print('Copied code+artifacts from', src)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 2. Configure paths

In [ ]:
import sys, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd

REPO = Path.cwd()
assert (REPO / 'experts').exists(), f'Run setup first - no experts/ in {REPO}'

def _find_data_root():
    hits = list(Path('/kaggle/input').rglob('dose_sys_train.csv'))
    if hits:
        return hits[0].parent.parent
    if (REPO / 'train' / 'dose_sys_train.csv').exists():
        return REPO
    return REPO  # experts reuse shipped CSVs; template comes from the repo copy

DATA_ROOT = _find_data_root()
_tmpl = DATA_ROOT / 'sample_submission.csv'
if not _tmpl.exists():
    _tmpl = REPO / 'sample_submission.csv'
TEMPLATE = pd.read_csv(_tmpl)
SCORING_CATEGORIES = ['all_drugs', 'all_opioids', 'all_stimulants']
EXPERTS = {n: REPO / 'experts' / n / 'run_expert.py'
           for n in ['lenny', 'william', 'jasmine', 'eddy']}
print(f'Repo: {REPO}')
print(f'Data: {DATA_ROOT}   Template: {len(TEMPLATE)} rows')

## 3. Run all four experts
Each expert runs in its own subprocess so their identically-named modules do not collide. If a shipped `expert_<name>.csv` is present it is reused as-is; otherwise that expert trains from the attached artifacts. A failing expert is reported and skipped, and the rest still combine.

In [ ]:
def run_expert(name, runner):
    out = REPO / f'expert_{name}.csv'
    if out.exists():
        print(f'\u2713 {name}: using shipped expert_{name}.csv (no retrain)')
        return True
    print(f'\n{"="*70}\n\u25b6 {name}\n{"="*70}', flush=True)
    t0 = time.time()
    p = subprocess.run(
        [sys.executable, str(runner), str(DATA_ROOT), f'expert_{name}.csv'],
        cwd=str(REPO), capture_output=True, text=True,
    )
    print(p.stdout[-4000:])
    if p.returncode != 0:
        print(f'\u2717 {name} FAILED (exit {p.returncode}):\n{p.stderr[-3000:]}')
        return False
    print(f'\u2713 {name} done in {time.time()-t0:.0f}s')
    return True

status = {n: run_expert(n, r) for n, r in EXPERTS.items()}
print('\nStatus:', status)

## 4. Combine experts → submission.csv

In [ ]:
preds = {}
for name in EXPERTS:
    f = REPO / f'expert_{name}.csv'
    if not f.exists():
        print(f'  {name}: MISSING'); continue
    preds[name] = (pd.read_csv(f).set_index('row_id')['rate_per_10000_ed_visits']
                   .reindex(TEMPLATE['row_id'].values))
available = list(preds)
assert available, 'No expert CSVs produced'
print('Combining:', available)

OOF_MAES = {
    'lenny':   {'all_drugs': 3.0651, 'all_opioids': 1.3563, 'all_stimulants': 0.6995},
    'william': {'all_drugs': 2.9066, 'all_opioids': 1.3319, 'all_stimulants': 0.6664},
    'jasmine': {'all_drugs': 3.0821, 'all_opioids': 1.3897, 'all_stimulants': 0.7217},
    'eddy':    {'all_drugs': 2.9711, 'all_opioids': 1.3628, 'all_stimulants': 0.6907},
}

final = pd.Series(0.0, index=TEMPLATE['row_id'])
for cat in SCORING_CATEGORIES:
    ids = TEMPLATE.loc[TEMPLATE['overdose_category'] == cat, 'row_id'].values
    mats, w = [], []
    for n in available:
        mats.append(preds[n].reindex(ids).fillna(preds[n].median()).values)
        mae = OOF_MAES.get(n, {}).get(cat)
        w.append(1.0 / mae if mae else 1.0)
    w = np.array(w); w = w / w.sum()
    final.loc[ids] = (np.column_stack(mats) @ w).clip(0)
    print(f'  {cat:15s} ' + '  '.join(f'{n}={wi:.3f}' for n, wi in zip(available, w)))

# Nesting: all_drugs >= all_opioids and >= all_stimulants
d = TEMPLATE.loc[TEMPLATE['overdose_category'] == 'all_drugs',      'row_id'].values
o = TEMPLATE.loc[TEMPLATE['overdose_category'] == 'all_opioids',    'row_id'].values
s = TEMPLATE.loc[TEMPLATE['overdose_category'] == 'all_stimulants', 'row_id'].values
final.loc[d] = np.maximum(np.maximum(final.loc[d].values, final.loc[o].values), final.loc[s].values)

submission = TEMPLATE[['row_id']].copy()
submission['rate_per_10000_ed_visits'] = final.reindex(TEMPLATE['row_id']).values
assert submission['rate_per_10000_ed_visits'].notna().all(), 'NaN in submission'
assert (submission['rate_per_10000_ed_visits'] >= 0).all(), 'Negative predictions'
assert set(submission['row_id']) == set(TEMPLATE['row_id']), 'row_id mismatch'

submission[['row_id', 'rate_per_10000_ed_visits']].to_csv('/kaggle/working/submission.csv', index=False)
print(f'\n\u2713 submission.csv written: {len(submission)} rows')
for cat in SCORING_CATEGORIES:
    v = submission.loc[(TEMPLATE['overdose_category'] == cat).values, 'rate_per_10000_ed_visits']
    print(f'  {cat:15s} mean={v.mean():.3f}  std={v.std():.3f}  min={v.min():.3f}  max={v.max():.3f}')
submission.head()